# Predicting 30-day Hospital Readmission for Diabetic Patients
By: Jose Fernando A. Gonzales

# About the Dataset

**Dataset name:** Diabetes 130-US hospitals for years 1999-2008 Data Set

**Background**

Diabetes Mellitus (DM) is a chronic disease where the blood has high sugar level. It can occur when the pancreas does not produce enough insulin, or when the body cannot effectively use the insulin it produces (WHO). Diabetes is a progressive disease that can lead to a significant number of health complications and profoundly reduce the quality of life. While many diabetic patients manage the health complication with diet and exercise, some require medications to control blood glucose level. As published by a research article named “The relationship between diabetes mellitus and 30-day readmission rates”, it is estimated that 9.3% of the population in the United States have diabetes mellitus (DM), 28% of which are undiagnosed. In recent years, government agencies and healthcare systems have increasingly focused on 30-day readmission rates to determine the complexity of their patient populations and to improve quality. Thirty-day readmission rates for hospitalized patients with DM are reported to be between 14.4 and 22.7%, much higher than the rate for all hospitalized patients (8.5–13.5%).

# Problem Statement

Hospitals aim to reduce avoidable 30-day readmissions because they increase healthcare costs and is an indicator of quality of care. This project uses machine learning to predict whether a diabetic patient will be readmitted within 30 days after discharge.

The project is primarily treated as a **binary classification problem**, where the target is whether a patient is readmitted within 30 days or not. Success is evaluated using technical metrics such as **AUC-ROC, recall, precision, F1-score, and confusion matrix analysis**, with emphasis on recall because failing to identify a high-risk patient may be more costly than a false positive.

## Business Goal
Support early identification of high-risk diabetic patients so hospitals can improve intervention planning, discharge support, and resource allocation.

## Machine Learning Goal
Develop and compare classification models that can accurately predict 30-day readmission risk.

## Notebook Purpose

This notebook implements the **Data Understanding** phase of the ML lifecycle for predicting 30-day hospital readmission of diabetic patients.

**Sections:**
1. **About the Dataset** — Background on the Diabetes 130-US Hospitals dataset and the clinical significance of 30-day readmission in diabetic populations
2. **Problem Statement** — Business and ML goals framed as a binary classification task prioritising recall
3. **Environment Setup** — Library imports and reproducibility settings
4. **Data Understanding** — Dataset loading, `df.info()`, descriptive statistics, and initial inspection
5. **Data Dictionary** — Detailed feature descriptions, types, and missingness for all 50 columns
6. **ID Mapping** — Reference tables for coded administrative fields (admission type, discharge disposition, admission source)

**ML Lifecycle Stage:** This is the first notebook in the project pipeline. Its outputs — the raw dataset saved to `data/raw/` and the structural/domain understanding documented here — feed directly into Notebook 02 (Preprocessing, EDA & Feature Engineering).

# Environment Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")
print(f"✓ Random seed set to {RANDOM_STATE}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")

✓ All libraries imported successfully!
✓ Random seed set to 42
✓ NumPy version: 2.4.3
✓ Pandas version: 3.0.1


# Data Understanding

## Importing the dataset

In [2]:
import kagglehub

path = kagglehub.dataset_download("saurabhtayal/diabetic-patients-readmission-prediction")

print("Path to dataset files:", path)

/Users/josefernandogonzales/Desktop/Solvin/Learning/Emeritus_AI/diabetic-readmission-prediction/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/josefernandogonzales/.cache/kagglehub/datasets/saurabhtayal/diabetic-patients-readmission-prediction/versions/2


In [3]:
df = pd.read_csv(f"{path}/diabetic_data.csv")

# Save the DataFrame to a CSV file for future use
data_raw_path = "../data/raw/diabetic_data.csv"
df.to_csv(data_raw_path, index=False)
print(f"✓ DataFrame saved to {data_raw_path}")

df.info()
display(df.describe())
display(df.head())

✓ DataFrame saved to ../data/raw/diabetic_data.csv
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 n

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Data Dictionary

| Feature name                | Type    | Description and values                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     | % missing |
| --------------------------- | ------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------: |
| Encounter ID                | Numeric | Unique identifier of an encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |        0% |
| Patient number              | Numeric | Unique identifier of a patient                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |        0% |
| Race                        | Nominal | Values: Caucasian, Asian, African American, Hispanic, and other                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |        2% |
| Gender                      | Nominal | Values: male, female, and unknown/invalid                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |        0% |
| Age                         | Nominal | Grouped in 10-year intervals: [0, 10), [10, 20), ..., [90, 100)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |        0% |
| Weight                      | Numeric | Weight in pounds                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           |       97% |
| Admission type              | Nominal | Integer identifier corresponding to 9 distinct values, for example, emergency, urgent, elective, newborn, and not available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |        0% |
| Discharge disposition       | Nominal | Integer identifier corresponding to 29 distinct values, for example, discharged to home, expired, and not available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |        0% |
| Admission source            | Nominal | Integer identifier corresponding to 21 distinct values, for example, physician referral, emergency room, and transfer from a hospital                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |        0% |
| Time in hospital            | Numeric | Integer number of days between admission and discharge                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     |        0% |
| Payer code                  | Nominal | Integer identifier corresponding to 23 distinct values, for example, Blue Cross\Blue Shield, Medicare, and self-pay                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |       52% |
| Medical specialty           | Nominal | Integer identifier of a specialty of the admitting physician, corresponding to 84 distinct values, for example, cardiology, internal medicine, family\general practice, and surgeon                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |       53% |
| Number of lab procedures    | Numeric | Number of lab tests performed during the encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |        0% |
| Number of procedures        | Numeric | Number of procedures (other than lab tests) performed during the encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |        0% |
| Number of medications       | Numeric | Number of distinct generic names administered during the encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |        0% |
| Number of outpatient visits | Numeric | Number of outpatient visits of the patient in the year preceding the encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |        0% |
| Number of emergency visits  | Numeric | Number of emergency visits of the patient in the year preceding the encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |        0% |
| Number of inpatient visits  | Numeric | Number of inpatient visits of the patient in the year preceding the encounter                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |        0% |
| Diagnosis 1                 | Nominal | The primary diagnosis (coded as first three digits of ICD9); 848 distinct values                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           |        0% |
| Diagnosis 2                 | Nominal | Secondary diagnosis (coded as first three digits of ICD9); 923 distinct values                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |        0% |
| Diagnosis 3                 | Nominal | Additional secondary diagnosis (coded as first three digits of ICD9); 954 distinct values                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |        1% |
| Number of diagnoses         | Numeric | Number of diagnoses entered to the system                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |        0% |
| Glucose serum test result   | Nominal | Indicates the range of the result or if the test was not taken. Values: `>200`, `>300`, `normal`, and `none` if not measured                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |        0% |
| A1c test result             | Nominal | Indicates the range of the result or if the test was not taken. Values: `>8` if the result was greater than 8%, `>7` if the result was greater than 7% but less than 8%, `normal` if the result was less than 7%, and `none` if not measured                                                                                                                                                                                                                                                                                                                                                                                                               |        0% |
| Change of medications       | Nominal | Indicates if there was a change in diabetic medications (either dosage or generic name). Values: `change` and `no change`                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |        0% |
| Diabetes medications        | Nominal | Indicates if there was any diabetic medication prescribed. Values: `yes` and `no`                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |        0% |
| 24 features for medications | Nominal | For the generic names: metformin, repaglinide, nateglinide, chlorpropamide, glimepiride, acetohexamide, glipizide, glyburide, tolbutamide, pioglitazone, rosiglitazone, acarbose, miglitol, troglitazone, tolazamide, examide, sitagliptin, insulin, glyburide-metformin, glipizide-metformin, glimepiride-pioglitazone, metformin-rosiglitazone, and metformin-pioglitazone, the feature indicates whether the drug was prescribed or there was a change in the dosage. Values: `up` if the dosage was increased during the encounter, `down` if the dosage was decreased, `steady` if the dosage did not change, and `no` if the drug was not prescribed |        0% |
| Readmitted                  | Nominal | Days to inpatient readmission. Values: `<30` if the patient was readmitted in less than 30 days, `>30` if the patient was readmitted in more than 30 days, and `No` for no record of readmission                                                                                                                                                                                                                                                                                                                                                                                                                                                           |        0% |


## ID Mapping

### Admission Type ID

| admission_type_id | description   |
| ----------------: | ------------- |
|                 1 | Emergency     |
|                 2 | Urgent        |
|                 3 | Elective      |
|                 4 | Newborn       |
|                 5 | Not Available |
|                 6 | NULL          |
|                 7 | Trauma Center |
|                 8 | Not Mapped    |

### Discharge Disposition ID

| discharge_disposition_id | description                                                                                               |
| -----------------------: | --------------------------------------------------------------------------------------------------------- |
|                        1 | Discharged to home                                                                                        |
|                        2 | Discharged/transferred to another short term hospital                                                     |
|                        3 | Discharged/transferred to SNF                                                                             |
|                        4 | Discharged/transferred to ICF                                                                             |
|                        5 | Discharged/transferred to another type of inpatient care institution                                      |
|                        6 | Discharged/transferred to home with home health service                                                   |
|                        7 | Left AMA                                                                                                  |
|                        8 | Discharged/transferred to home under care of Home IV provider                                             |
|                        9 | Admitted as an inpatient to this hospital                                                                 |
|                       10 | Neonate discharged to another hospital for neonatal aftercare                                             |
|                       11 | Expired                                                                                                   |
|                       12 | Still patient or expected to return for outpatient services                                               |
|                       13 | Hospice / home                                                                                            |
|                       14 | Hospice / medical facility                                                                                |
|                       15 | Discharged/transferred within this institution to Medicare approved swing bed                             |
|                       16 | Discharged/transferred/referred another institution for outpatient services                               |
|                       17 | Discharged/transferred/referred to this institution for outpatient services                               |
|                       18 | NULL                                                                                                      |
|                       19 | Expired at home. Medicaid only, hospice.                                                                  |
|                       20 | Expired in a medical facility. Medicaid only, hospice.                                                    |
|                       21 | Expired, place unknown. Medicaid only, hospice.                                                           |
|                       22 | Discharged/transferred to another rehab fac including rehab units of a hospital .                         |
|                       23 | Discharged/transferred to a long term care hospital.                                                      |
|                       24 | Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.   |
|                       25 | Not Mapped                                                                                                |
|                       26 | Unknown/Invalid                                                                                           |
|                       27 | Discharged/transferred to a federal health care facility.                                                 |
|                       28 | Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital |
|                       29 | Discharged/transferred to a Critical Access Hospital (CAH).                                               |
|                       30 | Discharged/transferred to another Type of Health Care Institution not Defined Elsewhere                   |


### Admission Source ID

| admission_source_id | description                                               |
| ------------------: | --------------------------------------------------------- |
|                   1 | Physician Referral                                        |
|                   2 | Clinic Referral                                           |
|                   3 | HMO Referral                                              |
|                   4 | Transfer from a hospital                                  |
|                   5 | Transfer from a Skilled Nursing Facility (SNF)            |
|                   6 | Transfer from another health care facility                |
|                   7 | Emergency Room                                            |
|                   8 | Court/Law Enforcement                                     |
|                   9 | Not Available                                             |
|                  10 | Transfer from critial access hospital                     |
|                  11 | Normal Delivery                                           |
|                  12 | Premature Delivery                                        |
|                  13 | Sick Baby                                                 |
|                  14 | Extramural Birth                                          |
|                  15 | Not Available                                             |
|                  16 | Not Available                                                      |
|                  17 | NULL                                                      |
|                  18 | Transfer From Another Home Health Agency                  |
|                  19 | Readmission to Same Home Health Agency                    |
|                  20 | Not Mapped                                                |
|                  21 | Unknown/Invalid                                           |
|                  22 | Transfer from hospital inpt/same fac reslt in a sep claim |
|                  23 | Born inside this hospital                                 |
|                  24 | Born outside this hospital                                |
|                  25 | Transfer from Ambulatory Surgery Center                   |
|                  26 | Transfer from Hospice                                     |
